# Dataset

In [ ]:
# INPUTS for  Dataset & Trainer

# file path ----------------------------------------------------------------------------------------------------------
data_dir = ''
raw_dir  = ''
sample_dir = '' # to save generated samples

# generate mdl dir --------------------------------------------------------------------------------------------------
mdl_results_parent = ''
milestone='20'
objective = 'pred_x0'

# testset sub ----------------------------------------------
sub_num = 1

target = []
target_arr = []

# ch_name like [37,31,6,15]
condition = [] 
cond_arr = []

rand_cond_flag = False # True: random condition will be applied

device_num = 0

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = str(device_num)

import torch
import numpy as np
import mne

import math
from scipy.integrate import cumtrapz

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import sys
import matplotlib

if sys.platform.startswith('win'):
    # Windows
    #matplotlib.use('Qt5Agg')
    %matplotlib qt
    print("Using qt backend")
else:
    # Linux / headless
    matplotlib.use('Agg')
    print("Using Agg backend")

import pickle

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # 명령 프롬트 창에 nvidia-smi 확인해서 설정 
device

In [ ]:
# load data

subs = {
    1:  {'name': '',    
         'bads': [], # bad channel name
         'depths': [], # non-ecog channel name
         'session': 0,  # session to use
         'drop_s1': [], # bad trial num
         'drop_s2': [], # bad trial num
         'notch': [], # notch filter (not used)
         'n_session1': 0}, # number of trials in session 1
}

sub_name = subs[sub_num]['name']
session_num = subs[sub_num]['session']

In [ ]:
def find_max_distance_timepoint(y, trig=100):
    # y shape: (3, seq_len, #trial)
    # trig   : timepoints of task trigger
    
    start_points = y[:, trig, :]  # Shape: (3, #trial)
    subsequences = y[:, trig:, :]  # Shape: (3, seq_len - trig, #trial)
    distances = np.linalg.norm(subsequences - start_points[:, np.newaxis, :], axis=0)  # Shape: (seq_len-trig, #trial)
    max_distance_indices = np.argmax(distances, axis=0)
    reach_times = max_distance_indices + trig

    return reach_times


# load data (2 session) and make event epochs (X, y)
def load_data(sub_num, X_mode, y_mode, dsfreq=100):
    
    # set parameters -------------------------------------------------------------------------------------------------------------------------------------------
    rmv = np.array([], dtype=int)  # save bad trial idx
    accumulated_trial = 0 # count total trial
    
    # nsteps: for sequence dataset
    tmin, tmax = 1, 3 # USER INPUT (!! tmin 도 양수여야함!)
    
    # initialize (self.X, self.y가 아님)
    # X data
    X_ME = np.zeros([0, 0, 0])
    X_MI = np.zeros([0, 0, 0])
    # y data
    y_vel = np.zeros([3, 0, 0])
    y_pos =  np.zeros([3, 0, 0])
    # sequence (4 direction label)
    sequences = np.zeros([0,])
        
    # 1. load all session data -----------------------------------------------------------------------------------------------------------------------------------
    for session in range(1, session_num + 1):
        
        # load data ---
        fname = f"{}.fif"
        epochs = mne.read_epochs(fname, preload=True)
        
        epochs_ME = epochs['ME']
        epochs_MI = epochs['MI']
        
        # load meta data ---
        fname = f"{}.npz"
        meta = np.load(fname)
        
        pos = meta['pos'] # (3, datapoint, trials)
        vel = meta['vel'] # (3, datapoint, trials)
        sequence = meta['sequence'] # (trial, 1)

        # load channel info ---
        fname = f''
        with open(fname, 'rb') as f:
            ch_info = pickle.load(f)
    
        # get parameters
        sfreq = epochs.info["sfreq"] # 2000 Hz
        s_ratio = int(sfreq/dsfreq)
    
        picks = mne.pick_types(epochs.info, meg=True, eeg=True, eog=True, stim=True, exclude='bads')
        nchan = len(picks)
    
        data_len = len(range(-np.int64(np.round(sfreq*tmin)), np.int64(np.round(sfreq*tmax)) + 1, s_ratio))
    
        # == TO REMOVE BAD TRIALS ==
        drop = f'drop_s{session}'
        drops =  subs[sub_num][drop]
    
        me_rmv = np.array([], dtype=int); mi_rmv = np.array([], dtype=int)
        for idx in drops: 
            n = idx // 2
            if idx % 2 == 0: me_rmv = np.append(me_rmv, n)    # ME              
            else: mi_rmv = np.append(mi_rmv, n) 
    
        if X_mode == 'ME': rmv = np.append(rmv, me_rmv + accumulated_trial)
        else: rmv = np.append(rmv, mi_rmv + accumulated_trial)

        # save sequence
        sequences = np.append(sequences, sequence)
    
        # Make X, y data -----------------------------------------------------------------------------------------------------------------------------
    
        # initialize
        if X_ME.shape[-1] == 0:
            X_ME = np.zeros([nchan, data_len, 0]); X_MI = np.zeros([nchan, data_len, 0])
            y_vel = np.zeros([3, data_len, 0]);    y_pos =  np.zeros([3, data_len, 0])
           
        # Downsampling: make X data (epochs, chan, datapoints) --> (chan, datapoitns, epochs) 
        data_ME = epochs_ME.get_data(copy=True) # (epochs, channels, time points)
        seg_ME = data_ME[:, :, ::s_ratio]
        seg_ME = np.transpose(seg_ME, (1, 2, 0)) # (chan, datapoints, epochs)
        X_ME = np.append(X_ME, seg_ME, axis = 2)
    
        data_MI = epochs_MI.get_data(copy=True)
        seg_MI = data_MI[:, :, ::s_ratio]
        seg_MI = np.transpose(seg_MI, (1, 2, 0))
        X_MI = np.append(X_MI, seg_MI, axis = 2)
        
        # Downsampling: make y data
        seg_vel = vel[:, ::s_ratio, :]
        y_vel = np.append(y_vel, seg_vel, axis =2)
    
        seg_pos = pos[:, ::s_ratio, :]
        y_pos = np.append(y_pos, seg_pos, axis =2)
    
        print('\n >> %d Events' %(X_ME.shape[-1])) # session 1 / session 2 Events
        accumulated_trial += X_ME.shape[-1]
    
                   
    # 2. set X, y ----------------------------------------------------------------------------------------------------------------------------------
    
    # Set data (remove bad trial idx for y data)
    if X_mode == 'ME': X = X_ME
    elif X_mode == 'MI': X = X_MI
    
    if y_mode == 'pos': y = y_pos; label = y_pos
    elif y_mode == 'vel': y = y_vel; label = y_pos

    # find time for reaching task
    reach_times = find_max_distance_timepoint(label, trig=100)
        
    X = np.delete(X, rmv, axis=2)
    y = np.delete(y, rmv, axis=2)
    label = np.delete(label, rmv, axis=2)
    sequences = np.delete(sequences, rmv) 
    reach_times = np.delete(reach_times, rmv)

    # FOR DIFFUSION ----------------------------------------------------
    X = np.transpose(X, (2, 0, 1)) 
    
    return X, y, label, sequences, reach_times, epochs, ch_info

In [ ]:
# load data ------------------------------------------------------------------------
X_mode = 'ME'
y_mode = 'vel'
dsfreq = 100

X, _, _, _, _, epochs, ch_info = load_data(sub_num, X_mode, y_mode, dsfreq)

### diffusion dataset

In [ ]:
from pathlib import Path
import json

# read config file -----------------------------------------------------
json_path = Path(mdl_results_parent) / 'config.json'

with json_path.open("r") as file:
    config = json.load(file)

train_session = config['train_session']
n_target_chan =  len(config['target'])
n_condition_chan = len(config['condition'])

In [ ]:
# intra
if train_session == 1 or train_session == 2:
    denorm_target_min = config['denorm_target_min']
    denorm_target_max = config['denorm_target_max']

# inter
else:
    min_val = np.min(X, axis=(0, 2), keepdims=False)
    max_val = np.max(X, axis=(0, 2), keepdims=False)
    mean_min = np.mean(min_val)
    mean_max = np.mean(max_val)
    denorm_target_min = np.full((n_target_chan,), mean_min)
    denorm_target_max = np.full((n_target_chan,), mean_max)

In [ ]:
from ecog_ddpm import Unet1D, GaussianDiffusion1D, Trainer1D, Dataset1D 

# make dataset for Diffusion Trainer

total_ch_names = epochs.info.ch_names

def get_chan_idx(channels, total_ch_names = total_ch_names):
    indices = []
    for ch in channels:
        if str(ch) in total_ch_names:
            indices.append(total_ch_names.index(str(ch)))
        else:
            print(f"Warning: Chan {ch} not found in total_ch_names.")
    return indices

target_idx = get_chan_idx(target_chan)
cond_idx = get_chan_idx(condition_chan)

# normalize 0 to 1 for each chan ---------------------------------------------------------------------------------
def normalize_channel_data(data, channels, rand_flag=False, mode='zero_to_one'):
    normalized_data = []
    min_vals = []
    max_vals = []

    assert mode in {'zero_to_one', 'minus_one_to_one'}, 'mode must be either "zero_to_one" or "minus_one_to_one"'

    data = data.astype(np.float32)

    if data.shape[-1] % 2 == 1:
        data = data[:, :, :-1]
    
    # normalize each channel
    for chan in channels:

        channel_data = data[:, chan:chan+1, :].astype(np.float32)
    
        min_val = np.min(channel_data, axis=(0, 2), keepdims=False)
        max_val = np.max(channel_data, axis=(0, 2), keepdims=False)
        
        # normalization; min-max scaling
        if mode == 'zero_to_one':
            channel_data_normalized = (channel_data - min_val) / (max_val - min_val)
        elif mode == 'minus_one_to_one':
            channel_data_normalized = 2 * (channel_data - min_val) / (max_val - min_val) - 1
        else:
            raise ValueError(f'unknown mode {mode}')
        
        normalized_data.append(channel_data_normalized)
        
        # save min, max
        min_vals.append(min_val.item())
        max_vals.append(max_val.item())
    
    # concat all chan
    normalized_data = np.concatenate(normalized_data, axis=1)
    normalized_tensor = torch.from_numpy(normalized_data)

    if rand_flag:
        if mode == 'zero_to_one':
            normalized_tensor  = torch.rand_like(normalized_tensor)
            print("=== !Condition Data Randomized (0 to 1)! ===\n")
        elif mode == 'minus_one_to_one':
            normalized_tensor = 2 * torch.rand_like(normalized_tensor) - 1
            print("=== !Condition Data Randomized (-1 to 1)! ===\n")
        else:
            raise ValueError(f'unknown mode {mode}')
    
    return normalized_tensor, min_vals, max_vals


# Testset ---------------------------------------------------------------------------------------
# train : using all data (for cross subjects)
if train_session == 0:

    # -- case when ground-truth target EXISTS --
    if target_idx:

        target_data, _, _ = normalize_channel_data(X, target_idx, False)
        condition_data, _, _ = normalize_channel_data(X, cond_idx, rand_cond_flag)
        
        train_trial, _, seq_len = target_data.shape
        train_trial, _, seq_len = condition_data.shape
    
        test_target = target_data; test_cond = condition_data
        testset = Dataset1D(target_data, condition_data)

    # -- case when ground-truth target does NOT EXIST --
    else:
        
        condition_data, _, _ = normalize_channel_data(X, cond_idx, rand_cond_flag)
        train_trial, n_condition_chan, seq_len = condition_data.shape

        # make empty target
        empty_target_data = torch.from_numpy(np.zeros([train_trial, n_target_chan, seq_len]))

        test_target = empty_target_data; test_cond = condition_data
        testset = Dataset1D(empty_target_data, condition_data)
    

# train: using 1 session data (for cross session / same subject)
else:
    # divide session 1 & 2
    n_session1 = subs[sub_num]['n_session1']
    X_s1 = X[:n_session1]
    X_s2 = X[n_session1:]

    # session 1
    target_s1_data, target_s1_min, target_s1_max = normalize_channel_data(X_s1, target_idx, False) # 무조건 false여야댐
    condition_s1_data, condition_s1_min, condition_s1_max = normalize_channel_data(X_s1, cond_idx, rand_cond_flag)
    
    # session 2
    target_s2_data, target_s2_min, target_s2_max = normalize_channel_data(X_s2, target_idx, False) # 무조건 false여야댐
    condition_s2_data, condition_s2_min, condition_s2_max = normalize_channel_data(X_s2, cond_idx, rand_cond_flag)
    
    if train_session == 1:
        
        train_trial, n_target_chan, seq_len = target_s1_data.shape
        train_trial, n_condition_chan, seq_len = condition_s1_data.shape     
        trainset = Dataset1D(target_s1_data, condition_s1_data)
       
        test_session = 2
        test_target = target_s2_data; test_cond = condition_s2_data
        test_trial, n_target_chan, seq_len = target_s2_data.shape
        test_trial, n_condition_chan, seq_len = condition_s2_data.shape
        testset = Dataset1D(target_s2_data, condition_s2_data)
        
    
    elif train_session == 2:
        
        train_trial, n_target_chan, seq_len = target_s2_data.shape
        train_trial, n_condition_chan, seq_len = condition_s2_data.shape     
        trainset = Dataset1D(target_s2_data, condition_s2_data)
    
        test_session = 1
        test_target = target_s1_data; test_cond = condition_s1_data
        test_trial, n_target_chan, seq_len = target_s1_data.shape
        test_trial, n_condition_chan, seq_len = condition_s1_data.shape
        testset = Dataset1D(target_s1_data, condition_s1_data)
    
    else:
        raise ValueError(f"Invalid train_session value: {train_session}. Must be either 0, 1, or 2.")

# Sampler: Generate data

In [ ]:
from pathlib import Path
import json
import shutil

# make sample dir -------------------------------------------------------------------------------------------
Path(sample_dir).mkdir(exist_ok=True)
# set sample dir

# INTRA
if target_chan:
    target_str = []
    cond_str = []
    
    sub_dir = Path('{}')
else:
    cond_str = []    
    sub_dir = Path('')


sub_dir.mkdir(parents=True ,exist_ok=True)
shutil.copy(json_path, os.path.join(sub_dir, 'used_mdl_info.json'))

In [ ]:
# save sample info

sample_info = {
    "sub_num": sub_num,
    "target": target_chan,
    "condition": condition_chan,
    "milestone": milestone,
    "rand_cond_flag": rand_cond_flag
}


sample_info_path = Path(sub_dir) / 'sample_info.json'
with sample_info_path.open('w') as file:
    json.dump(sample_info, file, indent=4)


### fn: Generate

In [ ]:
import torch
from ecog_ddpm import Unet1D, GaussianDiffusion1D, Trainer1D, Dataset1D 
from pathlib import Path

# generate train & test samles ----------------------------------------------------------------------------------
def generate_sample(dataset):
    real_x = dataset.x
    condition_x = dataset.condition_x
    samples = diffusion.sample(condition_x.to(device)).cpu()
    print(f'Generated shape: {samples.shape}') # (n_test_trial, #chan, seq_len)
    return samples

### generate all fold

In [ ]:
# file directory ---------------------------------------------------------------------------------------

print(f"Testset shape  | x: {testset.x.shape}  || condition_x: {testset.condition_x.shape}")

# load model -----------------------------------------------------------------------------
# Unet model ---
# [list] -> (tuple)
config["Unet_param"]["dim_mults"] = tuple(config["Unet_param"]["dim_mults"])
model = Unet1D(**config["Unet_param"])

# diffusion ---
diffusion = GaussianDiffusion1D(
    model,
    **config["diffusion_param"]
)

# Trainer ---
trainer = Trainer1D(
    diffusion,
    dataset = testset,
    **config["trainer_param"]
)

trainer.load(milestone)
device = trainer.device
diffusion.load_state_dict(trainer.model.state_dict())

# generate samples ----------------------------------------------------------------------------

print('>> Testset -------------------------')
generated_samples = generate_sample(testset)

print('\nGeneration completed')

### save generated samples

In [ ]:
# 1. denormalize 
def denormalize_data(data, min_vals, max_vals):
    denormalized_data = []

    channel_splits = torch.split(data, 1, dim=1)

    # Denormalize each channel
    for idx, chan_data in enumerate(channel_splits):
        min_val = min_vals[idx]
        max_val = max_vals[idx]
        
        # Denormalizing the data
        chan_data_denormalized = chan_data * (max_val - min_val) + min_val
        denormalized_data.append(chan_data_denormalized)

    # Concatenate all channels back together
    denormalized_data = torch.cat(denormalized_data, dim=1)

    return denormalized_data

denorm_generated_samples = denormalize_data(generated_samples, denorm_target_min, denorm_target_max)
print(f'denoramlized              || {denorm_generated_samples.shape}')

# 2. save dataset for decoder ----------------------------------
X_generated = denorm_generated_samples.numpy().transpose([1,2,0])
print(f'X_generated (to be saved) || {X_generated.shape}    (numpy)')    # (#trial, chan, datapoints) -> (chan, datapoints, #epochs)

# save numpy data
np.savez(f"{sub_dir}/samples-{milestone}.npz", X_generated=X_generated)
print('Generated testsets saved!')

## Results, Plot

In [ ]:
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

def calculate_pearsonr(x, y):
    return pearsonr(x, y)[0]

def calculate_rmse(x, y):
    return np.sqrt(mean_squared_error(x, y))

def calculate_results(target_data, condition_data, sample_data):
   
    n_trial = target_data.shape[0]
    
    # target vs generated sample
    tg_cc = np.zeros((n_trial, n_target_chan))
    tg_rmse = np.zeros((n_trial, n_target_chan))
    
    # target vs condition
    tc_cc = np.zeros((n_trial, n_target_chan, n_condition_chan))
    tc_rmse = np.zeros((n_trial, n_target_chan, n_condition_chan))
    
    # when ground-truth does EXIST
    if not torch.all(target_data == 0):
        # target vs generated sample
        for trial in range(n_trial):
            for channel in range(n_target_chan):
                tg_cc[trial, channel] = calculate_pearsonr(target_data[trial, channel].numpy(), sample_data[trial, channel].numpy())
                tg_rmse[trial, channel] = calculate_rmse(target_data[trial, channel].numpy(), sample_data[trial, channel].numpy())
        
        # target vs condition
        for trial in range(n_trial):
            for target in range(n_target_chan):
                for cond in range(n_condition_chan):
                    tc_cc[trial, target, cond] = calculate_pearsonr(target_data[trial, target].numpy(), condition_data[trial, cond].numpy())
                    tc_rmse[trial, target, cond] = calculate_rmse(target_data[trial, target].numpy(), condition_data[trial, cond].numpy())
   
    # when g.t does NOT exist
    else:
        pass
            
    # condition vs generated sample
    cg_cc = np.zeros((n_trial, n_target_chan, n_condition_chan))
    cg_rmse = np.zeros((n_trial, n_target_chan, n_condition_chan))
    for trial in range(n_trial):
        for generated in range(n_target_chan):
            for cond in range(n_condition_chan):
                cg_cc[trial, generated, cond] = calculate_pearsonr(sample_data[trial, generated].numpy(), condition_data[trial, cond].numpy())
                cg_rmse[trial, generated, cond] = calculate_rmse(sample_data[trial, generated].numpy(), condition_data[trial, cond].numpy())
    # 결과 출력
    print("Target-Generated CC Matrix:", tg_cc.shape)
    print("Target-Generated RMSE Matrix:", tg_rmse.shape)
    print("Target-Condition CC Matrix:", tc_cc.shape)
    print("Target-Condition RMSE Matrix:", tc_rmse.shape)
    print("Condition-Generated CC Matrix:", cg_cc.shape)
    print("Condition-Generated RMSE Matrix:", cg_rmse.shape)
    
    return {'tg_cc': tg_cc, 'tg_rmse': tg_rmse, 'tc_cc': tc_cc, 'tc_rmse': tc_rmse, 'cg_cc': cg_cc, 'cg_rmse': cg_rmse}

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

# Plot Results ----------------------------------------------------------------------------------------------------
# T: real_x (target), C: condition_x, G: samples (generated)     

fname = f"{sub_dir}/sub{sub_num}_{sub_name}_milestone-{milestone}.txt"

def prints(*args, **kwargs):
    with open(fname, "a") as f:
        print(*args, **kwargs)
        print(*args, **kwargs, file=f)

def plot_results(target_data, samples, results, plt_dir, condition_chan=condition_chan, target_chan=target_chan):
    
    # when target g.t not exists
    if not target_chan:
        target_chan_name = list(range(1, n_target_chan+1))
    else:
        target_chan_name = target_chan
    
    n_trial, n_target, _ = target_data.shape
    n_cond = len(condition_chan)
    
    for k in range(n_target):

        # target_dir = Path(f'{plt_dir}/{target_chan_name[k]}')
        # target_dir.mkdir(exist_ok=True)

        # for i in range(n_trial):
            
        #     fig, ax = plt.subplots(1, 1, figsize=(10, 3*2))
    
        #     target = target_data[i, k, :].numpy().flatten()
        #     generated = samples[i, k, :].numpy().flatten()
    
        #     #plot
        #     ax.plot(generated, label='Generated', color='firebrick', alpha=0.6)
        #     ax.plot(target, label='Target', color='royalblue', alpha=0.6)
        #     ax.set_title(f"Target (chan {target_chan_name[k]}) vs Generated (milestone: {milestone}) (CC: {results['tg_cc'][i, k]:.2f}, RMSE: {results['tg_rmse'][i, k]:.2f})")
    
        #     # plot
        #     fig.suptitle(f'Trial#{i} Target (chan {target_chan_name[k]}), Condition (# chan {n_condition_chan})\n', fontsize=18)
        #     plt.tight_layout()
        #     plt.savefig(f'{target_dir}/{target_chan_name[k]}_Trial-{i}.png', dpi=144) # , dpi=300
        #     plt.show()
        #     plt.close(fig)

        prints(f'---- Chan: {target_chan_name[k]} ----')
        avg_tg_cc = np.mean(results['tg_cc'][:,k], axis=0)
        avg_tg_rmse = np.mean(results['tg_rmse'][:,k], axis=0)
        avg_tc_cc = np.mean(results['tc_cc'][:,k,:], axis=0)
        avg_tc_rmse = np.mean(results['tc_rmse'][:,k,:], axis=0)
        avg_cg_cc = np.mean(results['cg_cc'][:,k,:], axis=0)
        avg_cg_rmse = np.mean(results['cg_rmse'][:,k,:], axis=0)
        
        prints(f"Target vs Generated |  cc: {avg_tg_cc:.5f}, rmse: {avg_tg_rmse:.5f}")
        for i in range(n_cond):
            prints(f"Generated vs Cond {condition_chan[i]} |  cc: {avg_cg_cc[i]:.5f}, rmse: {avg_cg_rmse[i]:.5f}")
        for i in range(n_cond):
            prints(f"Target vs Cond {condition_chan[i]} |  cc: {avg_tc_cc[i]:.5f}, rmse: {avg_tc_rmse[i]:.5f}")
        prints()

In [ ]:
# plot results --------------------------------------------------------------------------------
plt_dir = f'{sub_dir}/plots/milestone-{milestone}'
Path(plt_dir).mkdir(parents=True, exist_ok=True)

results = calculate_results(test_target, test_cond, generated_samples)

In [ ]:
plot_results(test_target, generated_samples, results, plt_dir)